# Large Earth Observation Models in Action

## Monitoring land use change nationally

Spatial Data Science Conference · New York · October 20, 2026 · Noam Rosenthal, LGND

An embedding model turns an image chip (about 150 m across in NAIP) into a vector of a few hundred numbers. Chips that look alike get vectors that are close together. These are embeddings of imagery, not of tabular attributes about a place like the PlacePulse model in the CARTO session.

LGND has computed and indexed them for every chip in two archives: NAIP aerial imagery of the contiguous US (1 m, 2020–2024) and Sentinel-2 satellite imagery of all land (10 m, monthly since 2017). Finding every subdivision under construction in the country, or every field that became one, is a query against that index rather than an image processing job.

The index is served as MCP tools at `https://geo.lgnd.ai/mcp`. Any MCP agent can use it; so can a Python script, which is where this notebook starts. The [`lgnd-geo-sdk`](https://github.com/EarthLegend/lgnd-geo-sdk) package handles sign-in and the connection.

The example is new housing on former farmland across the US: exposure for insurers, demand for utilities and telecom, pipeline for homebuilders and planners.

1. Connect
2. Scan the country for a feature and verify the matches against the imagery
3. Find where that change happened, when, and from what
4. Export to CARTO, your warehouse, or a desktop GIS
5. Hand the workflow to an LLM agent
6. An application: chat with a live map

You need an LGND account and, for sections 5 and 6, an Anthropic API key in `ANTHROPIC_API_KEY`. Free accounts get 150 tool calls a day. Sections 1 to 4 use about ten; each agent question uses five to thirty.

## 1. Connect

`client()` connects to the server. The first time, it opens a browser window to sign in or create an account, and saves the tokens in `~/.config/lgnd_geo_sdk/tokens`. `user_identity_get` returns who you're signed in as.

In [1]:
from lgnd_geo_sdk import client

async with client() as geo:
    me = await geo.call_tool("user_identity_get", {})

me.structured_content

{'user_id': 'usr_01a0f8e605c0768282cfe693616dc84e',
 'email': 'jeff@lgnd.ai',
 'name': 'Jeff Albrecht',
 'internal': True}

The server lists its tools, each with a description and an argument schema. That listing is the whole interface, for you and for the agent in section 5.

In [2]:
import pandas as pd

async with client() as geo:
    tools = await geo.list_tools()

pd.DataFrame([{"tool": t.name, "description": t.description.split(". ")[0]} for t in tools]).set_index("tool")

,description
tool,
user_identity_get,Return the platform user these tools act for: ...
user_alias_set,"Give a node, change or classifier a short name..."
search_place_find,"Find places that look like something, in eithe..."
search_change_find,Find places whose appearance changed from one ...
search_similar_change_find,Find observations or cells whose change looks ...
discover_area_view,Look at an area of one index at one level and ...
discover_area_shift_find,Rank an area's cells by how much they changed ...
discover_space_zoom,Move from a node through space
discover_time_zoom,Show one node through time: a cell period by p...


In [3]:
# One tool in full.
from geo_show import describe_tool

search = next(t for t in tools if t.name == "search_place_find")
describe_tool(search)

### `search_place_find`

Find places that look like something, in either index. Describe it in words with `text` (naip only), or give `like`: the id of an observation or cell that is the thing (the way to search s2). Returns up to `limit` matches, best first, each with its id, date or period, position and similarity. To label every place in an area instead, use toolbox_classifier_build and toolbox_classifier_apply.

For `text`, describe what an aerial photo of the thing shows, as a common noun: "solar farm", "hog farm", "charred forest". Avoid jargon, ground-level words ("wildfire") and words about time ("new"). Things smaller than about 10 m do not show on their own: describe their setting ("parking lot"). Try a few phrasings when it matters.

With `like`, level "observations" returns single images, one per place; a cell level returns cells in the node's own period. Takes a second or two.

| argument | | description |
|---|---|---|
| `index` | required | "naip" or "s2". |
| `text` |  | naip only: phrases for what an aerial photo shows, as common nouns ("solar farm"). Several are averaged. |
| `like` |  | The id (or name) of an observation or cell to resemble. |
| `area` |  | Where to look: a box {"south", "north", "west", "east"}, a circle {"lat", "lon", "radius_km"} or a polygon {"ring": [{"lat", "lon"}, ...]}, in degrees. Omit for the whole index. |
| `start` |  | Only observations on or after this date: YYYY, YYYY-MM or YYYY-MM-DD. |
| `end` |  | Only observations on or before this date: YYYY, YYYY-MM or YYYY-MM-DD. |
| `level` |  | With `like`: "observations" for single images, or a cell level number from the index's levels. Omit for the node's own kind. |
| `limit` |  | How many matches to return, 1-50. |

## 2. Scan the country for a feature

No LLM in this section or the next two. One tool call with no area, so the whole contiguous US. The description is embedded into the same space as the chips and the closest chips come back, one per location.

Each row is one observation: a chip at one location on one date. `similarity` ranks the matches; it is not a probability. Good matches on NAIP score around 0.3.

In [4]:
async with client() as geo:
    result = await geo.call_tool(
        "search_place_find",
        {"index": "naip", "text": "housing subdivision under construction", "limit": 20},  # no area: the whole index
    )

found = result.structured_content
print({k: v for k, v in found.items() if k != "matches"})  # the query, as the server ran it

matches = pd.DataFrame(found["matches"]["rows"], columns=found["matches"]["columns"])
matches

{'index': 'naip', 'like': "'housing subdivision under construction'", 'at': 'observations', 'where': 'everywhere'}


,node,place,observed,lat,lon,similarity
0,naip/obs:532118298,653fea9a23c2a1033a2b51f5cc857f2e,2021-11-04,34.8148,-87.6617,0.347
1,naip/obs:202392241,5b42845bb87789d642ed44b96f5e414f,2023-06-10,30.3261,-87.6590,0.341
2,naip/obs:17798560,73e9380561ed3f62131760301a4787b3,2022-07-10,41.0110,-74.0137,0.340
3,naip/obs:437485334,887a6c715a33cc528fedc88b5cd985a4,2021-11-26,34.7731,-86.9070,0.340
4,naip/obs:407200927,9fed3e8c1ebecd2bf71c805edc2c77a4,2023-05-25,36.1314,-86.5145,0.339
5,naip/obs:298928004,14092383a56b50c308fda807af9cb4b9,2021-10-14,34.9125,-111.8239,0.338
6,naip/obs:352832420,eac55d8a722f1b988e43a6c9bcf521a6,2023-08-09,40.3657,-74.4587,0.338
7,naip/obs:556007291,a02c56cd5f600facbcf92cd8c2d11e08,2023-07-14,42.9039,-112.4946,0.337
8,naip/obs:160685985,d01596a8f3a7ecdc44f506a7d716d966,2020-10-11,32.9894,-97.0472,0.337
9,naip/obs:120522116,8c8a69e212218f719ca13b01a7a0512c,2023-08-09,40.2866,-74.4788,0.337


In [5]:
from geo_show import show_map

show_map(found)

Map(center=[37.7735, -98.36285000000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_…

### Verify the matches against the imagery

A high score means the chip resembles the description, not that it is the thing: a subdivision under construction looks a lot like a cleared industrial site. The check fetches each chip's image and has an image model judge it against the claim. Nodes that pass are listed in `supported_at`; for the rest, it says what the image shows instead. Six nodes per call.

In [6]:
nodes = matches["node"].head(6).tolist()

async with client() as geo:
    result = await geo.call_tool(
        "reasoning_imagery_check",
        {
            "claim": "a residential subdivision under construction: graded lots, new streets, houses being built",
            "kind": "place",
            "nodes": nodes,
        },
    )

check = result.structured_content
check

{'verdict': 'supported',
 'supported_at': ['naip/obs:532118298',
  'naip/obs:202392241',
  'naip/obs:17798560',
  'naip/obs:437485334',
  'naip/obs:407200927',
  'naip/obs:298928004']}

In [7]:
verified = check.get("supported_at", [])
show_map(found, highlight=verified)  # green: verified

Map(center=[37.7735, -98.36285000000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_…

### Narrow to a region

The same search over the Phoenix metro, imagery from 2022 on. An area can also be a bounding box or a polygon ring.

In [8]:
PHOENIX = {"lat": 33.45, "lon": -112.07, "radius_km": 60}

async with client() as geo:
    result = await geo.call_tool(
        "search_place_find",
        {"index": "naip", "text": "housing subdivision under construction", "area": PHOENIX, "start": "2022", "limit": 10},
    )

nearby = result.structured_content
pd.DataFrame(nearby["matches"]["rows"], columns=nearby["matches"]["columns"])

,node,place,observed,lat,lon,similarity
0,naip/obs:299123094,0c5762b42fbb11ffbef5363ff9b11aae,2023-09-03,33.5155,-112.4779,0.324
1,naip/obs:406570287,d8c5cc4b597b495c28bd9d9e2474bb8a,2023-09-17,33.4724,-112.3805,0.322
2,naip/obs:489452997,1a135f8c134b292519ece9afcce028d4,2023-09-15,33.3042,-111.8708,0.322
3,naip/obs:127550667,e68d0dcfbeb8e6b0825055a065445544,2023-09-17,33.4925,-112.2811,0.320
4,naip/obs:178700333,786f084def50f8416fd6143ea9e11aa8,2023-09-17,33.5874,-112.4354,0.320
5,naip/obs:272398303,4465b5a09d313f95b27210bdc7618d11,2023-09-17,33.4379,-112.2928,0.320
6,naip/obs:592658324,c3e419429fd16b2fe51ac9fcddd294c8,2023-09-17,33.3703,-112.0986,0.318
7,naip/obs:275247073,76b67f66b046cb09df2c690f9dbd0fb7,2023-09-15,33.2237,-111.6325,0.318
8,naip/obs:677228949,1e2027b1bb1b90e2b9227a3847ff3bae,2023-09-17,33.4537,-112.3029,0.318
9,naip/obs:341578624,938de5003e1ed440c5b94f243d92ad27,2023-09-17,33.4911,-112.3375,0.317


In [9]:
show_map(nearby, area=PHOENIX)

Map(center=[33.405550000000005, -112.0552], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

## 3. Change detection: where, when, from what

Section 2 found subdivisions as they look now. This asks where a subdivision appeared where something else used to be. NAIP photographed most locations twice between 2020 and 2024, and the index holds the change between the two observations as well as the observations themselves. Each match is a change node, `naip/obs:12>obs:34`, with its two dates. Nationwide again.

In [10]:
async with client() as geo:
    result = await geo.call_tool(
        "search_change_find",
        {"index": "naip", "before": "farm fields", "after": "housing subdivision", "limit": 8},
    )

changes = result.structured_content
pairs = pd.DataFrame(changes["matches"]["rows"], columns=changes["matches"]["columns"])
pairs

,change,place,before,after,lat,lon,similarity
0,naip/obs:231391568>obs:89297225,2718dff8da67bf66f96e787996f2be13,2020-05-28,2022-05-24,38.3534,-122.6825,0.432
1,naip/obs:217254188>obs:613245853,dda9e47804bf27a906b9a3e40a3e22c6,2021-06-29,2023-06-29,46.2399,-119.9063,0.425
2,naip/obs:457872809>obs:40664917,c4378dea4a1bb416279e356c770a557f,2020-10-30,2022-09-13,29.8992,-95.7450,0.423
3,naip/obs:338570842>obs:53201138,3cb9d469d63dae2069241a2a08ed10d8,2020-05-28,2022-05-23,38.2916,-122.2737,0.422
4,naip/obs:626428469>obs:169954380,168d6ba87f3069708b47023b6ff28459,2020-10-30,2022-09-13,29.8144,-95.8441,0.422
5,naip/obs:173447562>obs:21024918,fd9c4a70548f65e59036c12f53a1dfa8,2020-10-30,2022-09-13,29.9036,-95.7374,0.421
6,naip/obs:560632563>obs:363060593,af110c4ec1dc1e3fdd1c56185a4ae57e,2020-06-11,2022-07-09,30.5432,-96.6743,0.421
7,naip/obs:514706652>obs:404271002,0585c77663cf73ccebb80a9681099bfa,2021-06-17,2023-07-12,39.0850,-77.2528,0.420


In [11]:
show_map(changes)

Map(center=[38.02715, -99.96764999999999], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in…

### When did it change

Every observation of the location in date order, with each one's similarity to the previous one. On NAIP that is two or three observations, enough to place the change between two dates; on Sentinel-2, with a monthly record, within a month.

In [12]:
top = pairs.iloc[0]
before_node = top["change"].split(">")[0]  # the observation before the change

async with client() as geo:
    result = await geo.call_tool("discover_time_zoom", {"node": before_node})

series = result.structured_content
print(f"{series['steps']} observations, {series['span'][0]} → {series['span'][1]}")
pd.DataFrame(series["rows"], columns=series["columns"])

2 observations, 2020-05-28 → 2022-05-24


,observed,node,to_previous,to_first_clear
0,2020-05-28,naip/obs:231391568,NaN,1.00
1,2022-05-24,naip/obs:89297225,0.76,0.76


### What changed

The same check on a change node: the image model sees the before and after images and judges whether they show the change described. This separates a land use change from a seasonal difference or a new crop.

In [13]:
print(f"{top['change']}   {top['before']} → {top['after']}   at {top['lat']}, {top['lon']}")

async with client() as geo:
    result = await geo.call_tool(
        "reasoning_imagery_check",
        {
            "claim": "open farm fields were replaced by a housing subdivision with streets and houses",
            "kind": "change",
            "nodes": [top["change"]],
        },
    )

result.structured_content

naip/obs:231391568>obs:89297225   2020-05-28 → 2022-05-24   at 38.3534, -122.6825


{'verdict': 'supported', 'supported_at': ['naip/obs:231391568>obs:89297225']}

## 4. Export to CARTO, your warehouse, or a desktop GIS

GeoJSON for the verified subdivisions, CSV with lat/lon for the change pairs. Both import into CARTO; the CSV loads into a BigQuery, Snowflake or Databricks table; both open in QGIS or ArcGIS. The node id on each record is the way back to the imagery and the input to any further tool call.

In [ ]:
import json

features = [
    {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [row["lon"], row["lat"]]},
        "properties": {
            "node": row["node"],
            "observed": row["observed"],
            "similarity": row["similarity"],
            "verified": row["node"] in verified,
        },
    }
    for row in matches.to_dict("records")
]
with open("subdivisions_under_construction.geojson", "w") as f:
    json.dump({"type": "FeatureCollection", "features": features}, f)

# lat/lon columns: imports into CARTO as points, loads into a warehouse table as is
pairs.to_csv("farmland_to_housing.csv", index=False)

print(f"subdivisions_under_construction.geojson: {len(features)} points, {len(verified)} verified")
print(f"farmland_to_housing.csv: {len(pairs)} change pairs")

## 5. Hand the workflow to an LLM agent

Sections 2 to 4 took a person choosing tools, writing descriptions and passing node ids between calls. An agent does that from a question. Here it is Claude through [Pydantic AI](https://pydantic.dev/docs/ai/); `toolset()` gives it the tool catalog from section 1 plus the server's instructions on how the tools fit together. Sections 2 and 3 as one question:

In [ ]:
import os
from getpass import getpass

from pydantic_ai import Agent

from lgnd_geo_sdk.pydantic_ai import toolset

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your ANTHROPIC_API_KEY: ").strip()

agent = Agent("anthropic:claude-opus-5-5", toolsets=[toolset()])
result = await agent.run(
    "Where in the US has farmland been converted to housing? Verify the five strongest cases against the imagery and give me their dates."
)
print(result.output)

### Watch the agent work

The run as a stream of events: thinking, each tool call, each result, the answer. Compare the agent's calls with yours from sections 2 and 3.

In [ ]:
from pydantic_ai.models.anthropic import AnthropicModelSettings

from geo_show import stream

# Return summarized thinking so it can be streamed.
THINKING = AnthropicModelSettings(anthropic_thinking={"type": "adaptive", "display": "summarized"})

agent = Agent("anthropic:claude-opus-5-5", toolsets=[toolset()], model_settings=THINKING)
run = await stream(agent, "Find subdivisions under construction within 60 km of Phoenix in imagery from 2022 or later, and verify the top three.")

### Tell the agent who it's working for

Instructions are where the agent is tailored to a job; `message_history` carries the conversation into the next question. Not shown: `toolset().filtered(...)` limits which tools it sees, and `.approval_required(...)` makes it ask before a call.

In [ ]:
INSTRUCTIONS = """\
I'm a planner tracking residential development across the US. Search the naip index. Leave the
area open unless I name a region.

Verify every finding with reasoning_imagery_check before reporting it. Report only the locations
it supported, and say how many it rejected and what they showed instead. Keep answers short: a
table of findings with the nearest town, the dates and coordinates, then a line of caveats."""

planner = Agent("anthropic:claude-opus-5-5", toolsets=[toolset()], instructions=INSTRUCTIONS, model_settings=THINKING)

run = await stream(planner, "Where around Phoenix has farmland become housing since 2021?")

In [ ]:
# A follow-up, in the same conversation.
run = await stream(planner, "Which of those are built out, and which are still under construction?", message_history=run.all_messages())

## 6. An application

Nothing new here: the section 5 agent with a chat panel and a live map, for someone who won't open a notebook. Each verification becomes a numbered pin, green where the imagery supported the claim and red where it didn't, and the answer cites the numbers.

`geo_demo.py` is about 500 lines of ipywidgets and ipyleaflet, no server. The part worth reading is `GeoChat._hook_toolsets`, which adds the finding number to each verification result before the model sees it; that is how citations match pins.

Click the four questions in order, or type your own.

In [ ]:
from geo_demo import GeoChat

APP_INSTRUCTIONS = INSTRUCTIONS + """

A map next to the chat shows the area and results of each tool call. Each imagery check becomes a
numbered finding on it; the check's result carries `finding`, the number. Cite findings in your
answer as [1], [2]. Name places rather than ids or coordinates, and do not mention embeddings,
similarity scores, node ids or tool names in your visible reasoning or your answer. Lead with
the answer in a sentence or two, then the evidence."""

DEMO = [
    "Where in the US is farmland being turned into housing? Show me the strongest cases.",
    "Take the strongest one: what was there before, what is there now, and between which dates did it change?",
    "Find the same kind of change within 60 km of Phoenix.",
    "Of the Phoenix cases, which are built out and which are still under construction?",
]

app = GeoChat(
    Agent("anthropic:claude-opus-5-5", toolsets=[toolset()], instructions=APP_INSTRUCTIONS, model_settings=THINKING),
    areas={"Phoenix": PHOENIX},
    examples=DEMO,
)
app

## Where to go from here

- **Other agent frameworks.** The server is standard MCP over HTTP with OAuth. `client()` and `auth()` work with any MCP client; LangGraph, the OpenAI Agents SDK or your own loop connect the same way. Pydantic AI is the adapter the SDK ships; for another, [open an issue](https://github.com/EarthLegend/lgnd-geo-sdk/issues).
- **No code.** Claude Code: `claude plugin marketplace add EarthLegend/skills`, then `claude plugin install lgnd-geo@lgnd`. Claude.ai, Claude Desktop or Cursor: add `https://geo.lgnd.ai/mcp` as an MCP server. The plugin's skills cover exhaustive search over an area with a classifier.
- **Your own area.** Sections 2 to 4 are a description, an area and a date range. Change those.
- **Docs and source:** [earthlegend.github.io/lgnd-geo-sdk](https://earthlegend.github.io/lgnd-geo-sdk/) · [github.com/EarthLegend/lgnd-geo-sdk](https://github.com/EarthLegend/lgnd-geo-sdk)